# Week 9a: LLM Evaluation — Metrics and Benchmarks
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand the evaluation landscape** — Why evaluation matters and the taxonomy of eval approaches
2. **Implement task-based evals** — Deterministic rules for length, keywords, format, and regex
3. **Apply reference-based metrics** — ROUGE and BERTScore, when each is appropriate
4. **Design custom evaluation suites** — Combine multiple metrics into a unified EvalSuite
5. **Connect evaluation to deployment decisions** — Failure thresholds and when metrics say "do not deploy"

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q openai rouge-score bert-score evaluate transformers pandas tqdm

In [ ]:
import os
import json
import re
import pandas as pd
from typing import List, Dict, Tuple, Optional
from getpass import getpass

from openai import OpenAI
from rouge_score import rouge_scorer
import evaluate
from tqdm.notebook import tqdm

### Sample Evaluation Data

We'll use a small dataset of prompts, references, and model outputs throughout this notebook.

In [ ]:
# Set OpenAI API key securely (works in Colab)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

---
## 1. The Evaluation Landscape

### Why Evaluation Matters

Evaluation is the **gatekeeper of deployment**. Without systematic evaluation:
- You cannot know if your model improved or regressed
- You risk shipping harmful, incorrect, or irrelevant outputs
- You lack evidence for stakeholders and regulators

### Taxonomy of Eval Approaches

| Approach | Description | Example |
|----------|-------------|--------|
| **Task-based** | Deterministic rules (length, keywords, format) | Length ≥ 20 words, contains "agent" |
| **Reference-based** | Compare output to gold reference | ROUGE, BERTScore |
| **Model-based** | Use another model to score | LLM-as-judge, rubric scoring |
| **Human** | Human annotators | A/B testing, preference surveys |

### Standard Benchmarks

- **MMLU** — Multi-task language understanding across 57 subjects
- **HellaSwag** — Commonsense reasoning
- **BIG-Bench** — Diverse tasks across 200+ benchmarks
- **HumanEval** — Code generation

We focus on **custom evals** for your specific use case.

---
## 2. Task-Based Evaluation

Task-based evals use **deterministic rules**:
- Length checks
- Keyword presence
- Format validation (e.g., JSON, bullet points)
- Regex patterns

In [ ]:
def task_eval(
    output: str,
    min_words: int = 20,
    required_keywords: List[str] = None,
    format_pattern: Optional[str] = None
) -> Tuple[int, List[str]]:
    """
    Evaluate output with deterministic rules.
    Returns (score, list of failure reasons).
    """
    score = 0
    reasons = []

    # Length check
    word_count = len(output.split())
    if word_count >= min_words:
        score += 1
    else:
        reasons.append(f"Too short ({word_count} words, need {min_words})")

    # Keyword presence
    if required_keywords:
        output_lower = output.lower()
        for kw in required_keywords:
            if kw.lower() in output_lower:
                score += 1
    else:
        score += 1  # No keywords required

    # Format validation (regex)
    if format_pattern:
        if re.search(format_pattern, output):
            score += 1
        else:
            reasons.append(f"Format validation failed (pattern: {format_pattern})")

    return score, reasons

# Example usage
output1 = "Reinforcement learning is about rewards. The agent takes actions and receives feedback."
output2 = "These models read a lot of text and learn patterns to generate predictions."

s1, r1 = task_eval(output1, min_words=20, required_keywords=["agent"])
s2, r2 = task_eval(output2, min_words=20, required_keywords=["agent"])

print(f"Output 1: score={s1}, reasons={r1}")
print(f"Output 2: score={s2}, reasons={r2}")

---
## 3. Reference-Based Metrics

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

- **ROUGE-1**: Unigram overlap
- **ROUGE-2**: Bigram overlap
- **ROUGE-L**: Longest common subsequence

**When to use**: Summarization, extraction tasks where lexical overlap matters.

### BERTScore

Uses contextual embeddings to match tokens semantically. Better for paraphrases and semantic similarity.

**When to use**: When references can be paraphrased; when semantic similarity matters more than exact words.

### Limitations

- **ROUGE**: Penalizes paraphrases; ignores semantics.
- **BERTScore**: Sensitive to embedding model; requires GPU for speed.

In [ ]:
data = [
    {
        "prompt": "Explain what reinforcement learning is in one paragraph.",
        "reference": "Reinforcement learning is a method where an agent learns by interacting with an environment and receiving rewards.",
        "student_output": "Reinforcement learning is about rewards. The agent takes actions and receives feedback."
    },
    {
        "prompt": "Summarize: Large language models are trained on massive text.",
        "reference": "LLMs learn statistical relationships from enormous datasets.",
        "student_output": "These models read a lot of text and learn patterns to generate predictions."
    },
]

df = pd.DataFrame(data)
df

In [ ]:
# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for _, row in df.iterrows():
    scores = scorer.score(row["student_output"], row["reference"])
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["rouge1"] = rouge1_scores
df["rouge2"] = rouge2_scores
df["rougeL"] = rougeL_scores
df[["student_output", "rouge1", "rouge2", "rougeL"]]

In [ ]:
# BERTScore
bertscore = evaluate.load("bertscore")
berts = bertscore.compute(
    predictions=df["student_output"].tolist(),
    references=df["reference"].tolist(),
    model_type="distilbert-base-uncased"
)
df["bertscore_f1"] = berts["f1"]
df[["student_output", "rougeL", "bertscore_f1"]]

---
## 4. Rubric-Based Evaluation (LLM-as-Scorer)

Use an LLM to score outputs on **clarity**, **correctness**, and **completeness** (1–5 scale).

In [ ]:
RUBRIC_PROMPT = """
You are an evaluator. Score the student answer on a scale of 1–5 for:

1. Clarity — Is the answer easy to understand?
2. Correctness — Does it match the reference?
3. Completeness — Does it cover key points?

Return JSON only: {"clarity": X, "correctness": Y, "completeness": Z}
"""

def rubric_eval(prompt: str, response: str, reference: str) -> str:
    message = f"""
PROMPT: {prompt}
REFERENCE: {reference}
STUDENT ANSWER: {response}
{RUBRIC_PROMPT}
"""
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}],
        temperature=0.0
    )
    return r.choices[0].message.content

df["rubric_scores"] = df.apply(
    lambda r: rubric_eval(r["prompt"], r["student_output"], r["reference"]),
    axis=1
)
df[["student_output", "rubric_scores"]]

---
## 5. Building Custom Eval Suites

Combine multiple metrics into a unified **EvalSuite** that runs all checks.

In [ ]:
class EvalSuite:
    """
    Runs multiple evaluation checks on model outputs.
    """
    def __init__(self, min_words: int = 20, required_keywords: List[str] = None):
        self.min_words = min_words
        self.required_keywords = required_keywords or []
        self.rouge_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        self.bertscore = evaluate.load("bertscore")

    def run_task(self, output: str) -> Tuple[int, List[str]]:
        return task_eval(output, self.min_words, self.required_keywords)

    def run_rouge(self, output: str, reference: str) -> float:
        return self.rouge_scorer.score(output, reference)["rougeL"].fmeasure

    def run_bertscore(self, outputs: List[str], references: List[str]) -> List[float]:
        result = self.bertscore.compute(
            predictions=outputs,
            references=references,
            model_type="distilbert-base-uncased"
        )
        return result["f1"]

    def run_all(self, data: List[Dict]) -> pd.DataFrame:
        """Run all eval checks on a list of {prompt, reference, student_output}."""
        rows = []
        for d in data:
            out = d["student_output"]
            ref = d["reference"]
            task_score, task_reasons = self.run_task(out)
            rouge_score = self.run_rouge(out, ref)
            rows.append({
                **d,
                "task_score": task_score,
                "task_reasons": task_reasons,
                "rougeL": rouge_score,
            })
        df = pd.DataFrame(rows)
        bert_f1 = self.run_bertscore(df["student_output"].tolist(), df["reference"].tolist())
        df["bertscore_f1"] = bert_f1
        return df


suite = EvalSuite(min_words=15, required_keywords=["agent", "reward"])
results = suite.run_all(data)
results

---
## 6. From Evaluation to Deployment Decisions

### Failure Thresholds

Define **when metrics say "do not deploy"**:

- Task score below threshold
- ROUGE-L below 0.2
- BERTScore F1 below 0.7
- Rubric average below 3.0
- Any safety flag

In [ ]:
def should_deploy(
    task_score: int,
    rougeL: float,
    bertscore_f1: float,
    task_threshold: int = 2,
    rouge_threshold: float = 0.2,
    bert_threshold: float = 0.7
) -> Tuple[bool, str]:
    """
    Returns (deploy_ok, reason).
    """
    if task_score < task_threshold:
        return False, f"Task score {task_score} below threshold {task_threshold}"
    if rougeL < rouge_threshold:
        return False, f"ROUGE-L {rougeL:.3f} below threshold {rouge_threshold}"
    if bertscore_f1 < bert_threshold:
        return False, f"BERTScore F1 {bertscore_f1:.3f} below threshold {bert_threshold}"
    return True, "All checks passed"

for _, row in results.iterrows():
    ok, reason = should_deploy(row["task_score"], row["rougeL"], row["bertscore_f1"])
    print(f"Deploy: {ok} — {reason}")

---
## 7. Exercises

1. **Task-based eval for a specific use case**: Design a task eval for a customer support chatbot that must (a) acknowledge the issue, (b) offer a solution, and (c) be under 150 words.

2. **Compare ROUGE vs BERTScore on 5 examples**: Create 5 (reference, output) pairs where one metric favors paraphrases and the other favors lexical overlap. Document your observations.

3. **Design an eval suite for a chatbot**: Define metrics (task + reference + rubric) and failure thresholds for a FAQ chatbot.

---
## 8. Responsible AI Reflection

**Evaluation blind spots**:

- Metrics can **overfit** to your eval set — diversify test cases.
- **ROUGE/BERTScore** ignore harm, bias, and factuality — combine with safety and rubric evals.
- **Rubric-based** evals inherit judge model biases — calibrate against human judgments.
- **Task-based** rules can be gamed — avoid overly narrow rules.

Always consider: *What would this metric miss?*

---
## Summary & Next Steps

You have learned:
- **Task-based** evals (deterministic rules)
- **Reference-based** metrics (ROUGE, BERTScore)
- **Rubric-based** evals (LLM-as-scorer)
- **EvalSuite** for combining metrics
- **Deployment thresholds** from evals

**Next**: Continue to **Week 9b** for **LLM-as-a-Judge** — pairwise comparisons, pointwise scoring, judge reliability, and calibration.